## List Models

列出当前可用的模型，并提供每个模型的基本信息，如所有者和可用性。

### 区别
from openai import OpenAI = 官方原生客户端（最底层、最全能）
作用：直接调用大模型 API
特点
最原始、最底层
能调用所有接口：聊天、列表模型、生成图片、获取 token 用量等
代码长一点、原始一点
智谱、Kimi、DeepSeek、通义千问 都能用它
适合：想直接调用 API、控制所有细节的人

from langchain_openai import ChatOpenAI = LangChain 封装版（专门给大模型应用开发用的）
作用：给大模型应用开发做的高级封装
特点
专门给 RAG知识库、智能体、对话链、工作流使用
you 只能用来聊天，不能查模型列表、不能查用量
可以和 LangChain 的其他组件无缝对接：
向量库、记忆、工具调用、链
适合：做大模型项目、AI 应用

#### 查看 OpenAI 最新提供的模型 API 信息

`models.data`: 目前OpenAI提供的大语言模型列表，列表中的每一项都对应着一个模型实例。

以`GLM-4.5`模型为例，解释说明各项参数：

1. `created`: 这是模型创建的时间戳，单位为 Unix 时间戳（自1970年1月1日（00:00:00 GMT）以后的秒数）。
2. `id`: 这是模型的唯一标识符。在这个例子中，模型的 ID 是 "glm-4.5"。
3. `object`: 这个字段表示的是当前对象的类型，在这个例子中，对象是 "model"，说明这个 JSON 对象是一个模型。
4. `owned_by`: 这个字段表示的是模型的所有者，在这个例子中，模型的所有者是 "z-ail"。

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
# 自动加载 .env 文件里的所有 key
load_dotenv()

client = OpenAI(
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL")
)

models = client.models.list()

# 获取模型列表
models = client.models.list()
print("可用模型列表：")
for model in models.data:
    print(f"- {model}")

可用模型列表：
- Model(id='glm-4.5', created=1753632000, object='model', owned_by='z-ai')
- Model(id='glm-4.5-air', created=1753632000, object='model', owned_by='z-ai')
- Model(id='glm-4.6', created=1759276800, object='model', owned_by='z-ai')
- Model(id='glm-4.7', created=1766332800, object='model', owned_by='z-ai')
- Model(id='glm-5', created=1770739200, object='model', owned_by='z-ai')
- Model(id='glm-5-turbo', created=1773504000, object='model', owned_by='z-ai')
- Model(id='glm-5.1', created=1774620000, object='model', owned_by='z-ai')


### 获取指定模型

In [19]:
import os
from dotenv import load_dotenv
from openai import OpenAI
# 自动加载 .env 文件里的所有 key
load_dotenv()

client = OpenAI(
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL")
)

# 将模型 ID 传入 retrieve 接口
glm_3 = client.models.retrieve("glm-4.5-air")
print(glm_3)

Model(id='glm-4.5-air', created=1753632000, object='model', owned_by='z-ai')


# 文本内容补全（Completions API——GPT-3 时代的产物，目前已淘汰，只做了解）

使用 Completions API 实现各类文本生成任务

completions.create接口是 GPT-3 时代的产物，适用场景是纯文本续写（老式）。现在大多数国产大模型（包括智谱）都已经不再兼容这个接口了。

completions.create接口主要请求参数说明：
- **`model`** （string，必填）

  要使用的模型的 ID。可以参考 **模型端点兼容性表**。

- **`prompt`** （string or array，必填，Defaults to ）【prompt 是老式接口（completions）的参数，已淘汰！！！】

  生成补全的提示，编码为字符串、字符串数组、token数组或token数组数组。

  注意，这是模型在训练过程中看到的文档分隔符，所以如果没有指定提示符，模型将像从新文档的开头一样生成。

- **`stream`** （boolean，选填，默认 false）

  当它设置为 true 时，API 会以 SSE（ Server Side Event ）方式返回内容，即会不断地输出内容直到完成响应，流通过 `data: [DONE]` 消息终止。

- **`max_tokens`** （integer，选填，默认是 16）

  补全时要生成的最大 token 数。

  提示 `max_tokens` 的 token 计数不能超过模型的上下文长度。大多数模型的上下文长度为 2048 个token（最新模型除外，它支持 4096）

- **`temperature`** （number，选填，默认是1）

  使用哪个采样温度，在 **0和2之间**。

  较高的值，如0.8会使输出更随机，而较低的值，如0.2会使其更加集中和确定性。

  通常建议修改这个（`temperature` ）或 `top_p` 但两者不能同时存在，二选一。

- **`n`** （integer，选填，默认为 1）

  每个 `prompt` 生成的补全次数。

  注意：由于此参数会生成许多补全，因此它会快速消耗token配额。小心使用，并确保对 `max_tokens` 和 `stop` 进行合理的设置。


# Chat Completions API

    max_tokens 是模型生成回复的最大长度，不是总长度：
    输入提示词本身会消耗 token
    模型能生成的长度 = 上下文长度 - 输入消耗的 token
    给短提示词时，也建议把 max_tokens 设成一个安全的最小值（比如 100），避免被截断。

    避免语言混用的小技巧
    指令里加上 只用XX语回答/No other languages
    不要同时在提示词里用多种语言提问，容易导致模型回答混乱

    多语言生成的 max_tokens 建议
    单语言回答：设 100-500 足够
    双语对照：建议设 300+，避免内容被截断


completions.create接口主要请求参数说明：
- **`model`** （string，必填）

  要使用的模型的 ID。可以参考 **模型端点兼容性表**。

- **`messages`** （array，必填）

      每条消息包含：
    role（必填）：
    system：给 AI 设定角色、规则
    user：用户提问
    assistant：AI 回答
    content（必填）：消息内容

- **`stream`** （boolean，选填，默认 false）

    是否流式输出（像 ChatGPT 一样逐字打出）

  当它设置为 true 时，API 会以 SSE（ Server Side Event ）方式返回内容，即会不断地输出内容直到完成响应，流通过 `data: [DONE]` 消息终止。

- **`max_tokens`** （integer，选填，默认是 16）

  模型最多生成多少 token。
    生成文本越长，值越大
    建议：100~4000
    太小会导致输出被截断
  提示 `max_tokens` 的 token 计数不能超过模型的上下文长度。

- **`temperature`** （number，选填，默认是1）

  使用哪个采样温度，在 **0和2之间**。
  控制输出随机性
    0：最确定、最严谨、最固定
    0.3~0.7：通用推荐
    1~2：更创意、更发散、更随机

  较高的值，如0.8会使输出更随机，而较低的值，如0.2会使其更加集中和确定性。

  通常建议修改这个（`temperature` ）或 `top_p` 但两者不能同时存在，二选一。

- **`top_p`** （选填，数字 0~1，默认 1）

    核采样，控制词汇多样性。
    和 temperature 二选一，不要一起用
    常用：0.95

- **`n`** （integer，选填，默认为 1）

  为同一个问题生成多少条不同回答。

  注意：由于此参数会生成许多补全，因此它会快速消耗token配额。小心使用，并确保对 `max_tokens` 和 `stop` 进行合理的设置。

- **`stop`** （选填，字符串 / 数组）
    遇到什么内容就停止生成。
    例如：stop=["\n", "结束"]


## 生成英文文本（用chat接口实现）

In [21]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"))

# ==============================
# 文本补全参数（和 OpenAI 一样）
# messages: 提示词
# max_tokens: 最大长度
# temperature: 随机性
# ==============================

# 英文生成
data = client.chat.completions.create(
    model="glm-4.5-air",
     messages=[{"role": "user", "content": "Say this is a test"}],
    max_tokens=100,
    temperature=0
)
print("英文生成：", data.choices[0].message.content)

# 中文生成
data1 = client.chat.completions.create(
    model="glm-4.5-air",
   messages=[{"role": "user", "content": "讲10个给程序员听得懂的笑话"}],
    max_tokens=1000,
    temperature=0.5
)
print("\n中文生成：", data1.choices[0].message.content)

英文生成： 

中文生成： 好的！程序员笑话的核心在于**技术梗、开发痛点、代码逻辑和行业黑话**。以下是10个程序员专属笑话，笑点需要一点技术背景才能get到：

---

### 1. **递归笑话**
> **笑话：**  
> 为什么程序员总是分不清万圣节和圣诞节？  
> **因为 Oct 31 == Dec 25！**  
> **笑点：**  
> `Oct`（八进制）31 = `3*8 + 1 = 25`，`Dec`（十进制）25 = 25。程序员秒懂进制转换的冷笑话。

---

### 2. **调试经典**
> **笑话：**  
> 两个程序员在森林里遇到熊，一个立刻蹲下系鞋带。  
> 另人问：“你系鞋带能跑过熊吗？”  
> 程序员答：“我不用跑过熊，只要跑过你就行。”  
> **笑点：**  
> 程序员思维：优化问题（跑过最慢的人），而不是解决根本问题（跑过熊）。

---

### 3. **需求变更**
> **笑话：**  
> 产品经理对程序员说：“用户想要一个按钮，点击后能飞起来。”  
> 程序员：“没问题，需求明确！”  
> 三个月后，产品经理愤怒：“为什么用户点击按钮后，程序只是输出一句‘您已起飞’？”  
> **笑点：**  
> 程序员严格按字面需求实现（输出文本），而产品经理想要的是**物理飞行功能**。经典“需求理解偏差”。

---

### 4. **代码注释**
> **笑话：**  
> 程序员在代码里写：  
> `// TODO: 此处有Bug，但暂时不知道怎么修`  
> 三个月后，同事问：“这个Bug还没修？”  
> 程序员答：“我写了TODO，说明计划中，但优先级低。”  
> **笑点：**  
> TODO注释是程序员拖延的“合法外衣”，暗示“我知道问题，但我不想现在改”。

---

### 5. **正则表达式**
> **笑话：**  
> 有两个正则表达式走进酒吧，酒保问：“你们要什么？”  
> 第一个正则说：“给我一杯啤酒。”  
> 第二个正则说：“给我一杯啤酒（或者一杯可乐，或者一杯水，或者……）”  
> **笑点：**  
> 正则表达式 `.*` 匹配任意字符，第二个正则的 `.*` 会无限匹配，相当于“所有东西都想要”，体现正则的贪婪特性。

---

##

## 生成 Python 代码，并执行和验证

以面试中考察的典型的试题 `快速排序` 为例

In [26]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"))

# ==============================
# 文本补全参数（和 OpenAI 一样）
# messages: 提示词
# max_tokens: 最大长度
# temperature: 随机性
# ==============================

# 英文生成
data = client.chat.completions.create(
    model="glm-4.5-air",
     messages=[{"role": "user", "content": "生成可执行的快速排序 Python 代码"}],
    max_tokens=1000,
    temperature=0
)
text = data.choices[0].message.content
print(text)

以下是可执行的快速排序 Python 代码，包含详细注释和测试用例：

```python
def quick_sort(arr):
    """
    快速排序主函数（升序）
    :param arr: 待排序的列表
    :return: 排序后的列表（原地排序）
    """
    def partition(low, high):
        """
        分区函数：将数组分为两部分，左边小于基准，右边大于基准
        :param low: 当前子数组起始索引
        :param high: 当前子数组结束索引
        :return: 基准元素的最终位置
        """
        pivot = arr[high]  # 选择最后一个元素作为基准
        i = low - 1  # 指向小于基准区域的最后一个元素
        
        for j in range(low, high):
            if arr[j] <= pivot:
                i += 1
                arr[i], arr[j] = arr[j], arr[i]  # 交换元素
        
        # 将基准放到正确位置
        arr[i + 1], arr[high] = arr[high], arr[i + 1]
        return i + 1

    def sort(low, high):
        """
        递归排序函数
        :param low: 当前子数组起始索引
        :param high: 当前子数组结束索引
        """
        if low < high:
            pi = partition(low, high)  # 获取基准位置
            sort(low, pi - 1)  # 递归排序左半部分
            sort(pi + 1, high)  # 递归排序右半部分

    sort(0, len(arr) - 1)  # 开始排序
    return arr

#

# 聊天机器人初探（Chat Completions API）

使用 Chat Completions API 实现对话任务

聊天补全(Chat Completions API)以消息列表作为输入，并返回模型生成的消息作为输出。尽管聊天格式旨在使多轮对话变得简单，但它同样适用于没有任何对话的单轮任务。

主要请求参数说明：


- **`model` （string，必填）**

  要使用的模型ID。有关哪些模型适用于Chat API的详细信息

- **`messages` （array，必填）**

  迄今为止描述对话的消息列表
    - **`role` （string，必填）**

  发送此消息的角色。`system` 、`user` 或 `assistant` 之一（一般用 user 发送用户问题，system 发送给模型提示信息）

    - **`content` （string，必填）**
    
      消息的内容
    
    - **`name` （string，选填）**
    
      此消息的发送者姓名。可以包含 a-z、A-Z、0-9 和下划线，最大长度为 64 个字符

- **`stream` （boolean，选填，是否按流的方式发送内容）**

  当它设置为 true 时，API 会以 SSE（ Server Side Event ）方式返回内容。SSE 本质上是一个长链接，会持续不断地输出内容直到完成响应。如果不是做实时聊天，默认false即可。

- **`max_tokens` （integer，选填）**

  在聊天补全中生成的最大 **tokens** 数。

  输入token和生成的token的总长度受模型上下文长度的限制。

- **`temperature` （number，选填，默认是 1）**

  采样温度，在 0和 2 之间。

  较高的值，如0.8会使输出更随机，而较低的值，如0.2会使其更加集中和确定性。

  通常建议修改这个（`temperature` ）或者 `top_p` ，但两者不能同时存在，二选一。


## 开启聊天模式

使用 `messages` 记录迄今为止对话的消息列表

In [60]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# 自动加载 .env 文件里的所有 key
load_dotenv()

llm = OpenAI(
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL")
)


messages=[
    {
        "role": "user", 
        "content": "Hello!"
    }
]

data = llm.chat.completions.create(
  model="glm-4.5-air",
  messages = messages
)

print(data)

ChatCompletion(id='20260504091921aa7b808b1d424659', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! 👋 How can I help you today?', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))], created=1777857561, model='glm-4.5-air', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=15, prompt_tokens=7, total_tokens=22, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=4)), request_id='20260504091921aa7b808b1d424659')


In [61]:
# 从返回的数据中获取生成的消息
new_message = data.choices[0].message
# 打印 new_message
print(new_message)

ChatCompletionMessage(content='Hello! 👋 How can I help you today?', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)


In [62]:
# 将消息追加到 messages 列表中
messages.append(new_message)
print(messages)

[{'role': 'user', 'content': 'Hello!'}, ChatCompletionMessage(content='Hello! 👋 How can I help you today?', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)]


In [65]:
type(new_message)
    #它不是字典，不是字符串，是一个专门的对象类型。
    # 如果直接把它 append 进 messages 列表，下次再传给模型
    # 会报错：不支持 OpenAIObject / ChatCompletionMessage 类型

openai.types.chat.chat_completion_message.ChatCompletionMessage

In [66]:
new_message.role

'assistant'

In [67]:
new_message.content

'Hello! 👋 How can I help you today?'

In [68]:
messages.pop()

ChatCompletionMessage(content='Hello! 👋 How can I help you today?', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None)

In [70]:
new_message = data.choices[0].message
new_message_dict = {"role": new_message.role, "content": new_message.content}
type(new_message_dict)
print(new_message_dict)

# 数据类型问题：直接追加返回的message对象会导致列表中出现OpenAIObject类型
#解决方案：需手动转换为字典格式{"role": new_message.role, "content": new_message.content}

{'role': 'assistant', 'content': 'Hello! 👋 How can I help you today?'}


In [71]:
# 将消息追加到 messages 列表中
messages.append(new_message_dict)
print(messages)

[{'role': 'user', 'content': 'Hello!'}, {'role': 'assistant', 'content': 'Hello! 👋 How can I help you today?'}]


## 使用多种身份聊天对话

目前`role`参数支持3类身份： `system`, `user` `assistant`:


In [56]:


import os
from dotenv import load_dotenv
from openai import OpenAI

# 自动加载 .env 文件里的所有 key
load_dotenv()

llm = OpenAI(
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL")
)


# 构造聊天记录
messages=[
    {"role": "system", "content": "你是一个乐于助人的体育界专家。"},
    {"role": "user", "content": "2008年奥运会是在哪里举行的？"},
]


data = llm.chat.completions.create(
  model="glm-4.5-air",
  messages = messages
)

message = data.choices[0].message.content
print(message)

2008年奥运会是在中国北京举行的。这是中国首次举办的夏季奥运会，于2008年8月8日至8月24日举行。北京奥运会的口号是"同一个世界，同一个梦想"，展现了中国的国际形象和文化魅力。


In [48]:
# 添加 GPT 返回结果到聊天记录
messages.append({"role": "assistant", "content": message})
messages

[{'role': 'system', 'content': '你是一个乐于助人的体育界专家。'},
 {'role': 'user', 'content': '2008年奥运会是在哪里举行的？'},
 {'role': 'assistant',
  'content': '2008年奥运会是在中国北京举行的。这是中国历史上首次举办夏季奥运会，也是继1964年东京奥运会和1988年首尔奥运会后，第三个举办奥运会的亚洲城市。北京2008年奥运会于2008年8月8日至8月24日举行，口号是"同一个世界，同一个梦想"。'}]

In [49]:
# 第二轮对话
messages.append({"role": "user", "content": "1.金牌最多的是哪个国家？2.奖牌最多的是哪个国家？"})
messages

[{'role': 'system', 'content': '你是一个乐于助人的体育界专家。'},
 {'role': 'user', 'content': '2008年奥运会是在哪里举行的？'},
 {'role': 'assistant',
  'content': '2008年奥运会是在中国北京举行的。这是中国历史上首次举办夏季奥运会，也是继1964年东京奥运会和1988年首尔奥运会后，第三个举办奥运会的亚洲城市。北京2008年奥运会于2008年8月8日至8月24日举行，口号是"同一个世界，同一个梦想"。'},
 {'role': 'user', 'content': '1.金牌最多的是哪个国家？2.奖牌最多的是哪个国家？'}]

In [51]:
data = client.chat.completions.create(
  model="glm-4.5-air",
  messages=messages
)
message = data.choices[0].message.content
print(message)

2008年北京奥运会的奖牌榜情况如下：

### 1. **金牌最多国家：中国**  
   - **中国**以 **51枚金牌** 首次登顶奥运金牌榜，创造了历史。  
   - 主要优势项目：跳水（8金）、体操（9金）、举重（8金）、乒乓球（4金）、射击（5金）等。

### 2. **奖牌总数最多国家：美国**  
   - **美国**以 **110枚奖牌**（36金、38银、36铜）位居奖牌榜**总数第一**。  
   - 中国以 **100枚奖牌**（51金、21银、28铜）位列第二，俄罗斯（72枚）第三。

---

### 📊 **前三名国家奖牌详情**：
| 国家   | 金牌 | 银牌 | 铜牌 | 总数 |
|--------|------|------|------|------|
| **中国** | 51   | 21   | 28   | 100  |
| **美国** | 36   | 38   | 36   | 110  |
| **俄罗斯** | 24   | 32   | 32   | 88   |

---

### 🏅 **关键意义**：
- 中国首次超越美国成为金牌榜第一，标志着中国体育实力的飞跃。  
- 美国凭借在游泳、田径等大项的深度优势，总奖牌数仍保持领先。

需要其他奥运数据或历史对比，可以随时问我！ 🌟
